# P2 — Chargement · Typage · Valeurs Impossibles · Plancher

**Sujet 05 — Santander Satisfaction**  
**Porteur P2** · Enchaînement : 1.2 → 1.3 → 1.5 → 1.10

| Section | Titre | Question clé |
|---|---|---|
| 1.2 | Charger et Compter | Combien de lignes, colonnes, mémoire ? |
| 1.3 | Typer | Quel type pour chaque colonne ? |
| 1.5 | Valeurs Impossibles | Y a-t-il des valeurs incohérentes ? |
| 1.10 | Le Plancher | Quel score minimal doit-on battre ? |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
# Localisation des données
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_05_Santander_Satisfaction"

def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / "data", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    essais = []
    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Données introuvables.\n"
        "Placez train.csv et a_decider.csv dans le même dossier que ce carnet.\n"
        "Chemins essayés : " + str(essais[:12])
    )

DOSSIER = trouver_donnees()
print("Données trouvées :", DOSSIER)

---
## 1.2 — Charger et Compter

In [ ]:
import time

t0 = time.time()
train = pd.read_csv(DOSSIER / "train.csv")
a_decider = pd.read_csv(DOSSIER / "a_decider.csv")
t1 = time.time()

print(f"Chargement en {t1-t0:.1f} secondes")
print()
print("=== COMPTAGE ===")
print(f"  train.csv     : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")
print(f"  a_decider.csv : {a_decider.shape[0]:,} lignes × {a_decider.shape[1]} colonnes")
print(f"  Mémoire train     : {train.memory_usage(deep=True).sum() / 1e6:.1f} Mo")
print(f"  Mémoire a_decider : {a_decider.memory_usage(deep=True).sum() / 1e6:.1f} Mo")

In [ ]:
# Valeurs manquantes (NaN)
nan_train = train.isna().sum().sum()
nan_decider = a_decider.isna().sum().sum()
print(f"Valeurs NaN dans train     : {nan_train}")
print(f"Valeurs NaN dans a_decider : {nan_decider}")
print("Note : les manquants sont codés (sentinelles), pas en NaN.")

---
## 1.3 — Typer

On recense le type de chaque colonne et on décide du typage définitif pour le pipeline.

In [ ]:
# Distribution des types bruts
print("Types détectés par pandas :")
print(train.dtypes.value_counts())
print()

# Colonnes non-numériques
non_num = train.select_dtypes(exclude='number').columns.tolist()
print(f"Colonnes non-numériques : {non_num if non_num else 'Aucune'}")

In [ ]:
# Identifier les colonnes binaires (0/1 uniquement)
features = [c for c in train.columns if c not in ('ID', 'TARGET')]
binaires = [c for c in features if set(train[c].unique()) <= {0, 1}]
continues = [c for c in features if c not in binaires and train[c].nunique() > 10]
discretes = [c for c in features if c not in binaires and train[c].nunique() <= 10]

print(f"Colonnes binaires  (0/1)         : {len(binaires)}")
print(f"Colonnes continues (>10 valeurs) : {len(continues)}")
print(f"Colonnes discrètes (≤10 valeurs) : {len(discretes)}")
print(f"  Exemples discrètes : {discretes[:5]}")

In [ ]:
# Résumé du typage
typage = pd.DataFrame({
    'colonne': features,
    'dtype_pandas': [str(train[c].dtype) for c in features],
    'n_uniques': [train[c].nunique() for c in features],
    'type_logique': [
        'binaire' if c in binaires
        else 'discrète' if c in discretes
        else 'continue'
        for c in features
    ]
})
print("Résumé du typage :")
print(typage['type_logique'].value_counts())
print()
display(typage.head(15))

---
## 1.5 — Valeurs Impossibles

Les valeurs **impossibles** ou **sentinelles** sont des valeurs numériques qui codent « information manquante » plutôt qu'une vraie mesure. Exemple classique : `-999999` dans `var3`.

In [ ]:
# Valeurs minimales extrêmes (candidates sentinelles négatives)
mins = train[features].min().sort_values()
print("Colonnes avec les valeurs minimales les plus basses :")
print(mins.head(10))

In [ ]:
# Vérification spécifique de var3 (sentinelle connue : -999999)
if 'var3' in train.columns:
    print("Distribution de var3 :")
    print(train['var3'].value_counts().head(10))
    n_sentinelle = (train['var3'] == -999999).sum()
    print(f"\nValeur sentinelle -999999 dans var3 : {n_sentinelle} lignes ({n_sentinelle/len(train)*100:.2f}%)")

In [ ]:
# Recherche de sentinelles génériques (valeurs très négatives ou très grandes)
SEUIL_SENTINELLE = -9999
sentinelles_trouvees = {}

for col in features:
    vals_suspectes = train[col][train[col] < SEUIL_SENTINELLE].unique()
    if len(vals_suspectes) > 0:
        sentinelles_trouvees[col] = vals_suspectes.tolist()

print(f"Colonnes avec valeurs < {SEUIL_SENTINELLE} (probables sentinelles) : {len(sentinelles_trouvees)}")
for col, vals in sentinelles_trouvees.items():
    n = (train[col] < SEUIL_SENTINELLE).sum()
    print(f"  {col}: valeurs={vals}, n={n}")

In [ ]:
# Vérification : valeurs négatives dans des colonnes de montants (impossibles par nature)
cols_montants = [c for c in features if c.startswith(('saldo_', 'imp_'))]
negatifs_montants = {}
for col in cols_montants:
    n_neg = (train[col] < 0).sum()
    if n_neg > 0:
        negatifs_montants[col] = n_neg

print(f"Colonnes de montants (saldo_, imp_) avec valeurs négatives : {len(negatifs_montants)}")
if negatifs_montants:
    for col, n in sorted(negatifs_montants.items(), key=lambda x: -x[1])[:10]:
        print(f"  {col}: {n} valeurs négatives")

In [ ]:
print("=== BILAN 1.5 — Valeurs impossibles ===")
print(f"  Sentinelles négatives trouvées : {len(sentinelles_trouvees)} colonnes")
print(f"  Montants négatifs suspects     : {len(negatifs_montants)} colonnes")
if 'var3' in train.columns:
    n_s = (train['var3'] == -999999).sum()
    print(f"  var3 (-999999)                 : {n_s} lignes → à remplacer par NaN ou médiane")

---
## 1.10 — Le Plancher

Le **plancher** est le score minimal qu'un modèle doit battre pour être utile. Il correspond à la performance de la **règle métier actuelle** de la banque (sans modèle ML).

D'après l'énoncé : **AUC plancher = 0,6890**

In [ ]:
from sklearn.metrics import roc_auc_score

CIBLE = "TARGET"
PLANCHER_AUC = 0.6890

y = train[CIBLE]

# Référence 1 : prédire toujours 0
auc_naif_zero = 0.5  # par définition, score constant → AUC = 0.5

# Référence 2 : utiliser une variable simple comme score brut (var15 souvent corrélée à l'âge)
if 'var15' in train.columns:
    auc_var15 = roc_auc_score(y, train['var15'])
    print(f"AUC avec var15 seul : {auc_var15:.4f}")

print(f"AUC modèle naïf     : {auc_naif_zero:.4f}")
print(f"Plancher métier     : {PLANCHER_AUC:.4f}  ← à battre absolument")

In [ ]:
# Visualisation du plancher
fig, ax = plt.subplots(figsize=(7, 4))

niveaux = {
    'Hasard (AUC=0.50)': 0.50,
    'Plancher métier (AUC=0.6890)': PLANCHER_AUC,
    'Objectif raisonnable (AUC=0.80)': 0.80,
    'Excellent (AUC=0.85)': 0.85,
}
couleurs = ['gray', 'orange', 'steelblue', 'green']

for (label, val), couleur in zip(niveaux.items(), couleurs):
    ax.barh(label, val, color=couleur, alpha=0.7)
    ax.text(val + 0.005, label, f'{val:.4f}', va='center', fontsize=9)

ax.set_xlim(0, 1)
ax.set_xlabel("AUC-ROC")
ax.set_title("Niveaux de performance — le plancher à battre")
plt.tight_layout()
plt.savefig("../sorties/plancher.png", dpi=100)
plt.show()
print("Figure sauvegardée dans sorties/plancher.png")

In [ ]:
print("=== BILAN 1.10 — Le Plancher ===")
print(f"  Plancher AUC à battre : {PLANCHER_AUC}")
print(f"  Référence hasard      : 0.5000")
print(f"  Marge à combler       : {PLANCHER_AUC - 0.5:.4f} points d'AUC au-dessus du hasard")
print()
print("  Un modèle qui fait AUC < 0.6890 est moins bon que la règle actuelle de la banque.")